# Approach A — Baseline, Ceiling, and the Matched-Size Control

Establishes the reference points every later approach is measured against.

| system | what it is |
|---|---|
| **baseline** | one classifier, all training rows — the floor |
| **ceiling** | one classifier per topic, that topic's rows only, routed by the **true** topic label |
| **matched** | one classifier per topic-sized **random** sample — same volume, different composition |

**Why three and not two.** A naive ceiling conflates two opposing effects: specialising
*helps*, but each specialist sees ~1/7 of the data, which *hurts*. On this corpus they
cancel almost exactly, so a two-system comparison reports ≈0 and looks like
"specialising doesn't work". The matched control separates them.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200)

from src import data, splits, diagnostics
from src.approaches import approach_a as A

## 1. The data, and the problem with it

FineFake, normalised to `text`, `label`, `topic`, `platform`, `n_words`.
`label` is **0 = fake, 1 = real** (verified against the fine-grained label column).

In [2]:
df = data.load_finefake()
print(f"rows: {len(df):,}   labels: {df['label'].value_counts().to_dict()}")
pd.DataFrame({
    "n": df["topic"].value_counts(),
    "pct_fake": (df.groupby("topic")["label"].apply(lambda s: 100*(s==0).mean())).round(1),
})

rows: 16,909   labels: {0: 9407, 1: 7502}


,n,pct_fake
topic,,
Business,1003,52.1
Conflict,1718,49.7
Entertainment,3699,58.9
Health,710,42.8
Politics,5727,51.9
Society,3939,63.7
Uncategorized,113,60.2


FineFake pools very different sources. Snopes entries are one-line claims;
AP News entries are full articles.

In [3]:
pd.DataFrame({
    "n": df["platform"].value_counts(),
    "median_words": df.groupby("platform")["n_words"].median().round(0),
    "pct_fake": (df.groupby("platform")["label"].apply(lambda s: 100*(s==0).mean())).round(1),
}).sort_values("n", ascending=False)

,n,median_words,pct_fake
platform,,,
snope,7556,18.0,67.3
reddit,4048,7.0,76.3
cnn,2310,266.0,10.3
washingtonpost,1041,1021.0,32.3
twitter,952,22.0,62.1
apnews,734,891.0,9.3
cdc_gov,268,244.0,1.1


**Short text is mostly fake; long text is mostly real.** A model can score well by
counting words rather than reading them — quantified in section 4.

## 2. Splits

Stratified jointly on `(label, topic)`, created once, committed to
`artifacts/splits.json`, and never regenerated. Every approach must run on identical
splits or the comparisons between them mean nothing.

In [4]:
sp = splits.load_splits()
splits.verify_splits(df, sp)        # raises if the data no longer matches
tr, va, te = splits.apply_splits(df, sp)
print({k: len(v) for k, v in sp.items()}, "— verified")

{'train': 11836, 'val': 2536, 'test': 2537} — verified


## 3. Run all three systems

In [5]:
res = A.run(df, sp, seed=42, write=False)
res[res["scope"] == "overall"][["system","n_train","n_test","macro_f1","roc_auc","mcc"]].round(4)

,system,n_train,n_test,macro_f1,roc_auc,mcc
0,baseline,11836,2537,0.7727,0.8506,0.5475
8,ceiling,11836,2520,0.7731,0.8466,0.5492


In [6]:
print(f"pooled ceiling - baseline = {A.gap(res):+.4f}   <- the naive number")

pooled ceiling - baseline = +0.0004   <- the naive number


Essentially zero. Taken alone this says specialising does nothing. It is misleading.

### The decomposition

- `specialisation` = ceiling − matched → what topic composition buys, **data volume held equal**
- `starvation` = matched − baseline → the price of training on less data
- `net` = ceiling − baseline → what a hard split actually delivers, and the sum of the other two

In [7]:
dec = A.decompose(res)
dec

,n_train,baseline,matched,ceiling,specialisation,starvation,net
scope,,,,,,,
Politics,4009.0,0.7706,0.7558,0.7629,0.0071,-0.0148,-0.0077
Society,2757.0,0.7689,0.7470,0.7679,0.0209,-0.0219,-0.0010
Entertainment,2589.0,0.8062,0.7770,0.8163,0.0394,-0.0293,0.0101
Conflict,1203.0,0.6911,0.6513,0.6993,0.0480,-0.0398,0.0082
Business,702.0,0.8000,0.7697,0.8092,0.0395,-0.0303,0.0092
Health,497.0,0.7154,0.7023,0.7447,0.0424,-0.0130,0.0294


In [8]:
print(f"mean specialisation = {dec['specialisation'].mean():+.4f}"
      f"   positive in {(dec['specialisation'] > 0).sum()}/{len(dec)} topics")
print(f"mean starvation     = {dec['starvation'].mean():+.4f}")
print(f"mean net            = {dec['net'].mean():+.4f}")

mean specialisation = +0.0329   positive in 6/6 topics
mean starvation     = -0.0249
mean net            = +0.0080


**Specialising helps in every topic (+0.033). Splitting the data costs −0.025.
They cancel.**

The naive ceiling was never a fair test — it charged specialisation the full price of
discarding 85% of each model's training data. This is also why the multi-domain
literature (MDFEND, M³FEND) *shares* parameters across domains rather than
partitioning: it keeps the gain without paying the cost. That is Approach C.

## 4. The confound

How much of the score survives without reading the text at all?

In [9]:
base_f1 = float(res[(res.scope=="overall") & (res.system=="baseline")]["macro_f1"].iloc[0])
diagnostics.report(tr, te, baseline_f1=base_f1)

--- text-free shortcut baselines ---


                  probe  macro_f1  roc_auc
         majority class    0.3574      NaN
  log(word count) alone    0.7226   0.7893
platform identity alone    0.7105   0.7458

  real baseline macro_f1 = 0.7727
  best text-free probe   = 0.7226
  contribution of text   = +0.0502

--- length signal within each topic ---


        topic  n_train  length_only_f1  corr_len_real
     Business      702          0.7874         0.6158
     Conflict     1203          0.6436         0.3893
Entertainment     2589          0.7505         0.5942
       Health      497          0.7252         0.5750
     Politics     4009          0.7050         0.4370
      Society     2757          0.7420         0.4886
Uncategorized       79          0.2273        -0.0019


`log(word count)` alone reaches ≈0.72 against a baseline of ≈0.77 — one feature, no
text. The coefficients confirm it: the strongest terms are a Reddit tag and stopwords,
not deception cues.

In [10]:
for term, weight in A.inspect_terms(res, 12):
    print(f"   {term:<22} {weight:+.3f}")

   psbattle               -8.422
   haff                   +8.323
   and                    +4.940
   in                     +3.907
   vs                     +2.964
   2021                   +2.777
   as                     +2.734
   said                   +2.635
   mr                     +2.603
   on                     +2.424
   of                     +2.353
   the                    +2.300


## 5. Is the gain about topic, or about platform?

Topic and platform are tangled — Health skews to `cdc_gov` (1.1% fake), Entertainment
to `snope`/`reddit` (67–76% fake). So a "topic specialist" might just be learning each
outlet's base rate.

`scripts/test_topic_vs_platform.py` re-runs the matched-size test **inside each
platform**, where there is no outlet to learn and far less length variation.

In [11]:
res_tp = pd.read_csv(REPO / "results" / "topic_vs_platform.csv")
res_tp

,platform,topic,n_train,n_test,pct_fake,specialist,matched,gain
0,snope,Business,385,85,77.9,0.5005,0.5248,-0.0243
1,snope,Conflict,738,147,61.8,0.4947,0.5385,-0.0437
2,snope,Entertainment,944,202,62.0,0.6117,0.5472,0.0645
3,snope,Health,262,65,69.5,0.6154,0.4415,0.1739
4,snope,Politics,1711,342,67.2,0.5893,0.5637,0.0256
5,snope,Society,1255,271,72.3,0.6282,0.5952,0.0330
6,reddit,Entertainment,1070,215,80.2,0.8414,0.8554,-0.0140
7,reddit,Politics,783,197,69.7,0.9406,0.8814,0.0592
8,reddit,Society,803,176,78.7,0.8658,0.8270,0.0388
9,cnn,Entertainment,343,79,2.6,0.4803,0.4732,0.0071


In [12]:
print(res_tp.groupby("platform").agg(
    cells=("gain","size"), mean_gain=("gain","mean"),
    wins=("gain", lambda s: int((s>0).sum()))).round(4).to_string())
print(f"\nOVERALL within-platform topic gain = {res_tp['gain'].mean():+.4f}"
      f"   positive in {(res_tp['gain']>0).sum()}/{len(res_tp)} cells")
print("across platforms (section 3) it was            +0.0329")

                cells  mean_gain  wins
platform                              
apnews              1     0.0000     0
cnn                 3     0.0035     2
reddit              3     0.0280     2
snope               6     0.0382     4
twitter             2     0.0918     2
washingtonpost      1     0.0949     1

OVERALL within-platform topic gain = +0.0376   positive in 11/16 cells
across platforms (section 3) it was            +0.0329


**The gain survives with platform held constant (+0.038) — slightly larger, in fact.**
So it is genuinely about topic, not an artifact of which outlet published the article.

Where it *doesn't* appear is informative: `cnn` (+0.004) and `apnews` (0.000) are
12.8% and 8.1% fake — so lopsided that every model collapses toward predicting "real"
and there is barely any fake news to specialise on. Where both classes are present —
snope, twitter, washingtonpost — specialisation delivers 4–17 points.

## 6. What Approach A establishes

**1. Specialists beat generalists.** +0.038 within platform, 11/16 cells; +0.033 across
topics, 6/6. The premise holds, and the magnitude matches the ~3 points MDFEND reports.

**2. Hard partitioning throws that gain away.** Net ≈ 0, because −0.025 of starvation
cancels it. Any approach that partitions the data must share parameters to avoid this.

**3. The corpus has a length/platform shortcut.** `log(word count)` alone reaches 0.72.
Every headline number from here on must be reported beside the text-free probes — a
score a word-counter can match is not a detection score.

### Carried forward to B, C and D

- Use `artifacts/splits.json` unchanged. Never regenerate it.
- Call `diagnostics.report()` alongside every result.
- The reference points are **baseline 0.7727** and the **specialisation effect +0.033** —
  not the pooled ceiling, which is confounded.
- Approach C (mixture-of-experts) is the direct response to finding 2: it is the
  architecture that captures specialisation without starving each expert.